# 📊 Лабораторна робота 1 — Exploratory Data Analysis (EDA)
## Twitter Sentiment Analysis: Hate Speech Detection

**Мета:** Провести первинний аналіз даних (EDA) для датасету Twitter Sentiment Analysis,
дослідити розподіл класів, характеристики тексту та підготувати дані до навчання моделі.

## 1. Імпорт бібліотек

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud
from collections import Counter
import re
import warnings

warnings.filterwarnings('ignore')

# Styling
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12

print('✅ Libraries loaded successfully!')

## 2. Завантаження даних

In [ ]:
df = pd.read_csv('../data/raw/twitter.csv')
print(f'Dataset shape: {df.shape}')
print(f'Columns: {list(df.columns)}')
df.head(10)

## 3. Огляд структури даних

In [ ]:
print('=== Info ===')
df.info()
print()
print('=== Describe ===')
df.describe()

In [ ]:
print('=== Missing Values ===')
missing = df.isnull().sum()
print(missing[missing > 0] if missing.sum() > 0 else 'No missing values! ✅')
print()
print('=== Duplicates ===')
print(f'Duplicate rows: {df.duplicated().sum()}')

## 4. Розподіл цільової змінної (Target Distribution)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart
label_counts = df['label'].value_counts()
colors = ['#2ecc71', '#e74c3c']
axes[0].bar(label_counts.index.astype(str), label_counts.values, color=colors, edgecolor='black', alpha=0.85)
axes[0].set_title('Class Distribution', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Label')
axes[0].set_ylabel('Count')
axes[0].set_xticks([0, 1])
axes[0].set_xticklabels(['Clean (0)', 'Hate Speech (1)'])
for i, v in enumerate(label_counts.values):
    axes[0].text(i, v + 200, str(v), ha='center', fontweight='bold')

# Pie chart
axes[1].pie(label_counts.values, labels=['Clean (0)', 'Hate Speech (1)'],
            autopct='%1.1f%%', colors=colors, startangle=90,
            explode=(0, 0.05), shadow=True)
axes[1].set_title('Class Proportion', fontsize=14, fontweight='bold')

plt.suptitle('Target Variable Distribution', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print(f'\nClass 0 (Clean):       {label_counts[0]:>6} ({label_counts[0]/len(df)*100:.1f}%)')
print(f'Class 1 (Hate Speech): {label_counts[1]:>6} ({label_counts[1]/len(df)*100:.1f}%)')
print(f'Imbalance ratio:       1:{label_counts[0]//label_counts[1]}')

## 5. Аналіз довжини твітів

In [ ]:
df['tweet_length'] = df['tweet'].apply(lambda x: len(str(x)))
df['word_count'] = df['tweet'].apply(lambda x: len(str(x).split()))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Tweet length distribution
for label, color, name in [(0, '#2ecc71', 'Clean'), (1, '#e74c3c', 'Hate Speech')]:
    subset = df[df['label'] == label]
    axes[0].hist(subset['tweet_length'], bins=50, alpha=0.6, color=color, label=name, edgecolor='black')
axes[0].set_title('Tweet Length Distribution (characters)', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Number of Characters')
axes[0].set_ylabel('Frequency')
axes[0].legend()

# Word count distribution
for label, color, name in [(0, '#2ecc71', 'Clean'), (1, '#e74c3c', 'Hate Speech')]:
    subset = df[df['label'] == label]
    axes[1].hist(subset['word_count'], bins=40, alpha=0.6, color=color, label=name, edgecolor='black')
axes[1].set_title('Word Count Distribution', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Number of Words')
axes[1].set_ylabel('Frequency')
axes[1].legend()

plt.tight_layout()
plt.show()

# Stats
print('\n📊 Tweet Length Statistics:')
print(df.groupby('label')[['tweet_length', 'word_count']].describe().round(1))

## 6. Передобробка тексту для аналізу

In [ ]:
def clean_tweet(text):
    """Clean tweet text for analysis."""
    if not isinstance(text, str):
        return ''
    text = text.lower()
    text = re.sub(r'http\S+|www\S+|https\S+', '', text)  # URLs
    text = re.sub(r'@\w+', '', text)  # mentions
    text = re.sub(r'#(\w+)', r'\1', text)  # hashtags
    text = re.sub(r'[^a-zA-Z\s]', '', text)  # special chars
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df['clean_tweet'] = df['tweet'].apply(clean_tweet)

print('✅ Text cleaned!')
print('\nOriginal vs Cleaned example:')
for i in range(3):
    print(f'  Original: {df["tweet"].iloc[i][:80]}...')
    print(f'  Cleaned:  {df["clean_tweet"].iloc[i][:80]}...')
    print()

## 7. Найпоширеніші слова (Top Words)

In [ ]:
from collections import Counter

def get_top_words(texts, n=20):
    """Get top N most frequent words."""
    all_words = ' '.join(texts).split()
    # Remove common stopwords
    stopwords = {'the', 'a', 'an', 'is', 'it', 'to', 'in', 'of', 'and', 'for',
                 'on', 'that', 'this', 'with', 'i', 'you', 'my', 'me', 'not',
                 'are', 'was', 'but', 'at', 'be', 'have', 'has', 'he', 'she',
                 'they', 'we', 'so', 'if', 'do', 'just', 'all', 'or', 'rt',
                 'im', 'its', 'ur', 'dont', 'no', 'get', 'like', 'can'}
    words = [w for w in all_words if w not in stopwords and len(w) > 2]
    return Counter(words).most_common(n)

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

for idx, (label, title, color) in enumerate([(0, 'Clean Tweets', '#2ecc71'), (1, 'Hate Speech Tweets', '#e74c3c')]):
    top_words = get_top_words(df[df['label'] == label]['clean_tweet'])
    words, counts = zip(*top_words)
    axes[idx].barh(range(len(words)), counts, color=color, edgecolor='black', alpha=0.85)
    axes[idx].set_yticks(range(len(words)))
    axes[idx].set_yticklabels(words)
    axes[idx].set_title(f'Top 20 Words — {title}', fontsize=13, fontweight='bold')
    axes[idx].set_xlabel('Frequency')
    axes[idx].invert_yaxis()

plt.tight_layout()
plt.show()

## 8. Word Clouds

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

for idx, (label, title, colormap) in enumerate([(0, 'Clean Tweets', 'Greens'), (1, 'Hate Speech', 'Reds')]):
    text = ' '.join(df[df['label'] == label]['clean_tweet'])
    wc = WordCloud(width=800, height=400, max_words=100, background_color='white',
                   colormap=colormap, contour_width=1, contour_color='black').generate(text)
    axes[idx].imshow(wc, interpolation='bilinear')
    axes[idx].set_title(f'Word Cloud — {title}', fontsize=14, fontweight='bold')
    axes[idx].axis('off')

plt.tight_layout()
plt.show()

## 9. Приклади твітів з кожного класу

In [ ]:
print('=' * 60)
print('🟢 CLEAN TWEETS (label = 0) — Random samples')
print('=' * 60)
for tweet in df[df['label'] == 0]['tweet'].sample(5, random_state=42).values:
    print(f'  • {tweet}')
print()

print('=' * 60)
print('🔴 HATE SPEECH TWEETS (label = 1) — Random samples')
print('=' * 60)
for tweet in df[df['label'] == 1]['tweet'].sample(5, random_state=42).values:
    print(f'  • {tweet}')

## 10. Кореляційний аналіз числових ознак

In [ ]:
# Create additional numeric features for correlation
df['has_mention'] = df['tweet'].apply(lambda x: 1 if '@' in str(x) else 0)
df['has_url'] = df['tweet'].apply(lambda x: 1 if 'http' in str(x) else 0)
df['has_hashtag'] = df['tweet'].apply(lambda x: 1 if '#' in str(x) else 0)
df['exclamation_count'] = df['tweet'].apply(lambda x: str(x).count('!'))

numeric_cols = ['label', 'tweet_length', 'word_count', 'has_mention', 'has_url', 'has_hashtag', 'exclamation_count']
corr_matrix = df[numeric_cols].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, cmap='RdBu_r', center=0,
            square=True, linewidths=0.5, fmt='.3f')
plt.title('Correlation Matrix', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 11. Висновки EDA

### Основні спостереження:
1. **Дисбаланс класів:** Датасет значно незбалансований — чистих твітів набагато більше, ніж hate speech.
2. **Довжина тексту:** Розподіл довжини твітів подібний для обох класів.
3. **Ключові слова:** Hate speech твіти містять характерну лексику, яку модель може використати для класифікації.
4. **Дані потребують:** Очистку тексту, видалення URLs, mentions, спеціальних символів.

### Рекомендації для моделювання:
- Використати **F1-score** як основну метрику (через дисбаланс класів)
- Застосувати **TF-IDF** для перетворення тексту в числові ознаки
- Спробувати різні моделі: Logistic Regression, Random Forest, SVM